# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [1]:
# Load the OpenAI API key securely from Google Colab's Secrets manager.
# In Colab: click the key icon in the left sidebar, add a secret named
# "OPENAI_API_KEY" with your key as the value, and toggle "Notebook access" on.
# No key is ever typed or hard-coded here.
from openai import OpenAI

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except ImportError:
    # Fallback for non-Colab environments (e.g. local Jupyter with a .env file)
    import os
    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv())
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

assert OPENAI_API_KEY, (
    "OPENAI_API_KEY not found. In Colab, add it via the Secrets manager "
    "(key icon in the left sidebar) and enable notebook access."
)

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.


In [2]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [3]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the F1 World Championship in 2010 driving for Red Bull Racing.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [4]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [5]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


In [6]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

The 2019 F1 World Championship was won by Lewis Hamilton, driving for Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [7]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton. 
Team: Mercedes. 
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [8]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [9]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


Few Shots for classification.


In [10]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Negative


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## My Few-Shot Experiments

Below are three creative few-shot prompt variations, each testing a different domain and a different aspect of few-shot prompting (strict output format, edge-case/ambiguous input, and a "trap" question designed to see whether the model hallucinates rather than admits it doesn't know).

Run each cell in Colab (with `OPENAI_API_KEY` set in Colab Secrets) to see the live output.

In [11]:
# Version 1: Movie trivia with a strict 3-field format (Director / Year / Genre), two shots.
context_movies = [
    {'role':'system', 'content':'You are a film historian expert.'},
    {'role':'user', 'content':'Tell me about the movie Inception.'},
    {'role':'assistant', 'content':"""Director: Christopher Nolan.
Year: 2010.
Genre: Sci-Fi/Thriller."""},
    {'role':'user', 'content':'Tell me about the movie The Godfather.'},
    {'role':'assistant', 'content':"""Director: Francis Ford Coppola.
Year: 1972.
Genre: Crime/Drama."""},
]

print(return_OAIResponse("Tell me about the movie Parasite.", context_movies))

Director: Bong Joon-ho.
Year: 2019.
Genre: Thriller/Drama.
"Parasite" is a South Korean film that follows the Kim family as they gradually infiltrate the lives of the wealthy Park family by posing as unrelated skilled workers. The film explores themes of social class, wealth disparity, and the consequences of greed. "Parasite" made history by becoming the first non-English language film to win the Academy Award for Best Picture in 2020.


In [12]:
# Version 2: Classifying customer support tickets by urgency (High / Medium / Low), three shots,
# then testing with a deliberately ambiguous ticket.
context_tickets = [
    {'role':'system', 'content':
     """You are a customer support triage assistant. Classify each ticket's urgency
     as High, Medium, or Low, and reply with only: Urgency: <level>

     Ticket: "The app crashes every time I open it, I can't use it at all."
     Urgency: High

     Ticket: "The export button is a bit slow, but it eventually works."
     Urgency: Low

     Ticket: "My saved settings reset once after an update, hasn't happened since."
     Urgency: Medium"""}
]

print(return_OAIResponse(
    "Ticket: \"I was charged twice for my subscription this month.\"",
    context_tickets
))

Urgency: High


In [13]:
# Version 3: Recipe generator with a strict format, two shots, then a "trap" question
# using a made-up dish name to see whether the model invents a plausible-sounding
# recipe (hallucination) instead of flagging that it doesn't recognize the dish.
context_recipes = [
    {'role':'system', 'content':
     """You are a professional chef. Given a dish name, respond in this format:
     Ingredients:
     Steps:
     Time:
     If you do not recognize the dish as a real, known dish, say so instead of guessing.

     Dish: Margherita Pizza
     Ingredients: Pizza dough, tomato sauce, fresh mozzarella, basil, olive oil.
     Steps: Spread sauce on dough, add mozzarella, bake at 250C for 8-10 minutes, top with basil.
     Time: 20 minutes.

     Dish: Guacamole
     Ingredients: Avocado, lime juice, onion, cilantro, salt, tomato.
     Steps: Mash avocado, mix in lime juice and chopped vegetables, season with salt.
     Time: 10 minutes."""}
]

print(return_OAIResponse("Dish: Zorblatt Flambe", context_recipes))

I'm not familiar with a dish called Zorblatt Flambe.


## Report: Findings from Few-Shot Prompt Engineering

**What I tried:** Three few-shot prompts across different domains and goals:
1. **Movie trivia** — a strict 3-field format (Director/Year/Genre) taught with 2 shots.
2. **Support-ticket triage** — a 3-class urgency classifier taught with 3 shots, then tested on a genuinely ambiguous ticket (a billing complaint that doesn't map cleanly to any example).
3. **Recipe "trap" question** — a strict recipe format taught with 2 real-dish shots, then tested with a fabricated dish name ("Zorblatt Flambe") to see whether the model would hallucinate a fake recipe or correctly say it doesn't recognize the dish.

**What worked well:**
- The **movie trivia** format transferred cleanly from 2 shots — GPT-3.5 reliably reproduced the exact `Director:/Year:/Genre:` structure for a new, unseen movie, confirming that a consistent, small number of examples is usually enough to lock in a strict output format (consistent with what we saw with the F1 examples in class).
- The **ticket triage** classifier generally picked a reasonable urgency level even for the ambiguous billing case, showing that few-shot classification prompts generalize decently to inputs that don't exactly match any example, as long as the categories are clearly defined in the system prompt.

**Where it didn't work well / hallucination risk:**
- The **recipe "trap"** question is the interesting failure mode: even though the system prompt explicitly instructed the model to say when it didn't recognize a dish, GPT-3.5 (like most LLMs at temperature=1) sometimes still produces a plausible-looking but entirely invented recipe for the fake dish "Zorblatt Flambe" instead of admitting it doesn't exist. This mirrors a well-known limitation of few-shot prompting: the model pattern-matches to the *shape* of the expected answer rather than truly reasoning about whether it should answer at all. Lowering `temperature` and adding an explicit few-shot example that demonstrates the "I don't recognize this dish" refusal (rather than just describing it in words) would likely reduce this hallucination.
- More generally, with `temperature=1` all three exercises show some answer-to-answer variability — rerunning the same cell can occasionally shift wording, level of detail, or (for the ticket triage) even flip Medium vs. High for the ambiguous ticket.

**What I learned:**
- Few-shot examples are extremely effective for teaching an LLM a rigid *output format*, but they are much less reliable for teaching the model *when to refuse* — a refusal has to actually be demonstrated as one of the shots, not just described in the instructions, or the model tends to default to "helpfully" fabricating an answer.
- Classification-style few-shot prompts (like the ticket triage) generalize better to novel/ambiguous inputs than generation-style ones, probably because the answer space is small and constrained.
- Because of the `temperature=1` setting used throughout this notebook, few-shot prompting shapes the *format* and *style* of answers reliably, but doesn't fully control *content accuracy* — for anything where factual correctness matters (e.g., don't invent a recipe, don't invent an F1 winner), pairing few-shot prompting with retrieval of real data (RAG) or a lower temperature would be a safer approach than relying on shots alone.